# ATLANTIS for Relation Extraction — Flan-T5 Pipeline Test


## Three-model setup 

| Role | Symbol | Model | Notes |
|------|--------|-------|-------|
| Small base (frozen) | p^S_b | `flan-t5-small` (80M) | Un-finetuned, frozen |
| Reference model | p_r | `flan-t5-small` fine-tuned | Same arch as p^S_b, finetuned on seed/train data |
| Large model (trained) | p^L_b | `flan-t5-base` (250M) | Trained with ATLANTIS weights |



## Note on T5 vs the paper

The ATLANTIS paper uses decoder-only models. T5 is encoder-decoder, so the log-prob
computation is slightly different — we use the seq2seq loss directly rather than a
unified autoregressive chain. 

---
**Runtime**: About 30 minutes on A100

---
# Part 1 — Setup

## 1.1 Installation

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate sentencepiece
print("Done.")

## 1.2 Imports

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5ForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1.3 Config

In [ ]:
# ── Models ────────────────────────────────────────────────────────────────────
SMALL_MODEL = "google/flan-t5-small"   # backbone for p^S_b (frozen) and p_r (finetuned)
LARGE_MODEL = "google/flan-t5-base"    # p^L_b: frozen for weight pre-computation,
                                       #        then trained fresh as p_θ

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET = "semeval"

# ── Training ──────────────────────────────────────────────────────────────────
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 16
BATCH_SIZE     = 32
NUM_EPOCHS     = 8
PR_EPOCHS      = 8    
WARMUP_RATIO   = 0.1
LR             = 2e-4


LOGPROB_MODE = "mean"   

MODEL_TYPE = "seq2seq"  

WEIGHT_CLIP = None   

# ── Quick test mode ───────────────────────────────────────────────────────────
QUICK_TEST = False

if QUICK_TEST:
    NUM_EPOCHS = 1
    PR_EPOCHS  = 3
    SUBSAMPLE  = 0.05
    print("QUICK_TEST mode: 1 epoch, 5% data")
else:
    SUBSAMPLE = 1.0
    print("Full experiment mode")

print(f"Config set. LOGPROB_MODE={LOGPROB_MODE!r}  PR_EPOCHS={PR_EPOCHS}")
print()
print("ATLANTIS weight formula (Algorithm 1, pre-computed once before training):")
print("  W_i = p^L_b(y_i|x_i) / p^S_b(y_i|x_i)  *  p_r(y_i|x_i)")
print("  All three terms use FROZEN models — weights are constants during training.")

## 1.4 Dataset Loading

In [ ]:
# SemEval 2010 Task 8 has 19 classes:
# IDs 0-17: 9 directional relation pairs (e.g. Cause-Effect(e1,e2) and Cause-Effect(e2,e1))
# ID 18: Other
# We collapse directions → 10 canonical labels

SEMEVAL_CANONICAL = [
    "Cause-Effect",        # 0, 1
    "Component-Whole",     # 2, 3
    "Content-Container",   # 4, 5
    "Entity-Destination",  # 6, 7
    "Entity-Origin",       # 8, 9
    "Instrument-Agency",   # 10, 11
    "Member-Collection",   # 12, 13
    "Message-Topic",       # 14, 15
    "Product-Producer",    # 16, 17
    "Other",               # 18
]

CONLL_RELATIONS = ["no_relation", "Work-For", "Kill", "OrgBased-In", "Live-In", "Located-In"]

def semeval_label_to_str(label_id):
    if label_id == 18:
        return "Other"
    return SEMEVAL_CANONICAL[label_id // 2]

def load_examples(dataset_name, subsample=1.0):
    if dataset_name == "semeval":
        raw = load_dataset("sem_eval_2010_task_8")
        label_list = SEMEVAL_CANONICAL[:]
        def to_ex(item):
            return {
                "input_text": (
                    "Classify the semantic relation between the marked entities "
                    f"in this sentence: {item['sentence'].strip()}"
                ),
                "label_str": semeval_label_to_str(item["relation"]),
            }
        train = [to_ex(x) for x in raw["train"]]
        test  = [to_ex(x) for x in raw["test"]]

    elif dataset_name == "conll2004":
        raw = load_dataset("tner/conll2004")
        label_list = sorted(set(CONLL_RELATIONS))
        def to_ex(item):
            sentence = " ".join(item["tokens"])
            tags  = item.get("relations", [])
            label = (CONLL_RELATIONS[tags[0]["type"]]
                     if tags and isinstance(tags[0], dict) else "no_relation")
            return {
                "input_text": f"Classify the relation between entities: {sentence}",
                "label_str":  label,
            }
        train = [to_ex(x) for x in raw["train"]]
        test  = [to_ex(x) for x in raw["test"]]
    else:
        raise ValueError(f"Unknown dataset: {dataset_name!r}")

    if subsample < 1.0:
        n = max(100, int(len(train) * subsample))
        train = random.sample(train, n)

    return train, test, label_list

train_all, test_examples, LABEL_LIST = load_examples(DATASET, subsample=SUBSAMPLE)
print(f"Dataset:  {DATASET}")
print(f"Train:    {len(train_all)} | Test: {len(test_examples)}")
print(f"Labels ({len(LABEL_LIST)}): {LABEL_LIST}")

# Verify label mapping
if DATASET == "semeval":
    raw_check = load_dataset("sem_eval_2010_task_8")
    print("\nLabel mapping verification:")
    for i, name in enumerate(raw_check["train"].features["relation"].names):
        print(f"  {i:2d}: {name:35s} → {semeval_label_to_str(i)}")

## 1.5 Tokenizer

In [ ]:
# flan-t5-small and flan-t5-base share the same tokenizer
tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

## 1.6 Dataset Class & Data Loaders

In [ ]:
# Build label↔id mappings once, used by classification head
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

class REDataset(Dataset):
    """RE dataset supporting both seq2seq and classification MODEL_TYPE.

    seq2seq:        labels = tokenized target string with -100 padding mask
    classification: labels = integer class id (scalar)
    """
    def __init__(self, examples, label_source="gold", include_weights=False):
        self.examples        = examples
        self.label_source    = label_source
        self.include_weights = include_weights

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex    = self.examples[idx]
        label = ex["label_str"] if self.label_source == "gold" else ex["weak_label"]

        enc = tokenizer(
            ex["input_text"],
            max_length=MAX_INPUT_LEN, padding="max_length",
            truncation=True, return_tensors="pt",
        )

        if MODEL_TYPE == "classification":
            item = {
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         torch.tensor(LABEL2ID[label], dtype=torch.long),
            }
        else:
            tgt = tokenizer(
                label,
                max_length=MAX_TARGET_LEN, padding="max_length",
                truncation=True, return_tensors="pt",
            )
            labels = tgt["input_ids"].squeeze()
            labels[labels == tokenizer.pad_token_id] = -100
            item = {
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         labels,
            }

        if self.include_weights:
            item["weight"] = torch.tensor(ex["weight"], dtype=torch.float32)
        return item


def make_loaders(train_exs, label_source="gold", include_weights=False):
    train_ds = REDataset(train_exs, label_source=label_source,
                         include_weights=include_weights)
    test_ds  = REDataset(test_examples, label_source="gold")
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0),
        DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    )

print(f"Dataset class defined. MODEL_TYPE={MODEL_TYPE!r}")
print(f"LABEL2ID: {LABEL2ID}")

## 1.7 Core Utilities: Logprob and evaluation

In [ ]:
_snap_stats = {"exact": 0, "substring": 0, "logprob": 0}


def compute_seq_logprob(model, input_text, label_str):
    """log p(label_str | input_text) — branches on MODEL_TYPE."""
    enc = tokenizer(
        input_text, return_tensors="pt",
        max_length=MAX_INPUT_LEN, truncation=True,
    ).to(DEVICE)

    if MODEL_TYPE == "classification":
        label_id = LABEL2ID[label_str]
        with torch.no_grad():
            out = model(**enc)
        return torch.log_softmax(out.logits, dim=-1)[0, label_id].item()

    else:
        tgt = tokenizer(
            label_str, return_tensors="pt",
            max_length=MAX_TARGET_LEN, truncation=True,
        ).to(DEVICE)
        labels = tgt["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            out = model(**enc, labels=labels)

        mean_logprob = -out.loss.item()
        if LOGPROB_MODE == "sum":
            n_tokens = (labels != -100).sum().item()
            return mean_logprob * n_tokens
        else:
            return mean_logprob


def _snap_single(pred):
    """Try exact then substring match. Returns label or None."""
    if pred in LABEL_LIST:
        return pred
    for label in LABEL_LIST:
        if label.lower() in pred.lower() or pred.lower() in label.lower():
            return label
    return None


def _logprob_snap_batch(model, input_texts):
    """Score all valid labels for each input, return best label per input.
    Uses batched forward passes — one pass per label over all unresolved inputs.
    """
    results = [None] * len(input_texts)
    best_lp = [float("-inf")] * len(input_texts)

    for label in LABEL_LIST:
        # Batch all inputs against this one label
        enc = tokenizer(
            input_texts,
            return_tensors="pt",
            max_length=MAX_INPUT_LEN,
            padding=True,
            truncation=True,
        ).to(DEVICE)
        tgt = tokenizer(
            [label] * len(input_texts),
            return_tensors="pt",
            max_length=MAX_TARGET_LEN,
            padding=True,
            truncation=True,
        ).to(DEVICE)

        labels = tgt["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            # Run per-example to get individual losses, not batch mean
            for i, (iid, amask) in enumerate(
                zip(enc["input_ids"], enc["attention_mask"])
            ):
                out = model(
                    input_ids=iid.unsqueeze(0),
                    attention_mask=amask.unsqueeze(0),
                    labels=labels[i].unsqueeze(0),
                )
                lp = -out.loss.item()
                if LOGPROB_MODE == "sum":
                    lp *= (labels[i] != -100).sum().item()
                if lp > best_lp[i]:
                    best_lp[i] = lp
                    results[i] = label

    return results


def print_snap_stats(reset=True):
    total = sum(_snap_stats.values()) or 1
    print("Label-snapping stats:")
    for tier, count in _snap_stats.items():
        print(f"  {tier:<12} {count:>5}  ({count/total:.1%})")
    if _snap_stats["logprob"] / total > 0.05:
        print("  ⚠ >5% fell through to log-prob — check generation quality")
    if reset:
        for k in _snap_stats:
            _snap_stats[k] = 0


def evaluate(model, loader):
    """Generate predictions and return (macro-F1, preds, golds).

    Hybrid batching strategy for seq2seq:
      - Tier 1/2 (exact/substring): batched generate() over full batch
      - Tier 3 (log-prob):          batched per-label scoring, only for
                                    the subset that didn't resolve in T1/T2
    Classification uses a single batched argmax — no snapping needed.
    """
    model.eval()
    preds, golds = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            if MODEL_TYPE == "classification":
                out         = model(input_ids=input_ids, attention_mask=attention_mask)
                pred_ids    = out.logits.argmax(dim=-1).cpu().tolist()
                batch_preds = [ID2LABEL[i] for i in pred_ids]
                batch_golds = [ID2LABEL[i] for i in batch["labels"].cpu().tolist()]

            else:
                # ── Tier 1 + 2: batched generate then snap ────────────────────
                gen = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=MAX_TARGET_LEN,
                    num_beams=1,
                )
                decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)

                batch_preds = [None] * len(decoded)
                tier3_indices = []   # indices that need log-prob fallback
                tier3_texts   = []   # corresponding input texts for Tier 3

                decoded_inputs = tokenizer.batch_decode(
                    input_ids.cpu(), skip_special_tokens=True
                )

                for i, pred in enumerate(decoded):
                    snapped = _snap_single(pred.strip())
                    if snapped is not None:
                        batch_preds[i] = snapped
                        _snap_stats["exact" if pred.strip() in LABEL_LIST
                                    else "substring"] += 1
                    else:
                        tier3_indices.append(i)
                        tier3_texts.append(decoded_inputs[i])

                # ── Tier 3: batched log-prob scoring for unresolved examples ──
                if tier3_indices:
                    _snap_stats["logprob"] += len(tier3_indices)
                    tier3_preds = _logprob_snap_batch(model, tier3_texts)
                    for i, pred in zip(tier3_indices, tier3_preds):
                        batch_preds[i] = pred

                # ── Gold labels ───────────────────────────────────────────────
                lbl = batch["labels"].clone()
                lbl[lbl == -100] = tokenizer.pad_token_id
                batch_golds = [
                    t.strip() for t in
                    tokenizer.batch_decode(lbl.cpu(), skip_special_tokens=True)
                ]

            preds.extend(batch_preds)
            golds.extend(batch_golds)

    macro_f1 = f1_score(golds, preds, labels=LABEL_LIST, average="macro", zero_division=0)
    print_snap_stats(reset=True)
    return macro_f1, preds, golds


print(f"Utilities defined. MODEL_TYPE={MODEL_TYPE!r}  LOGPROB_MODE={LOGPROB_MODE!r}")
print("Label snapping: exact → substring → batched log-prob scoring")

## 1.8 ATLANTIS Loss


In [ ]:
def compute_per_example_ce(model, input_ids, attention_mask, labels):
    """Per-example mean CE — branches on MODEL_TYPE.

    seq2seq:        mean CE over decoder output tokens
    classification: CE over single class label (scalar per example)
    """
    if MODEL_TYPE == "classification":
        out    = model(input_ids=input_ids, attention_mask=attention_mask)
        ce     = nn.CrossEntropyLoss(reduction="none")
        return ce(out.logits, labels)   # (B,)

    else:  # seq2seq
        out     = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        B, T, V = out.logits.shape
        ce      = nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        per_tok = ce(out.logits.view(B*T, V), labels.view(B*T)).view(B, T)
        n_tok   = (labels != -100).float().sum(dim=-1).clamp(min=1)
        return per_tok.sum(dim=-1) / n_tok   # (B,)


def atlantis_loss(model, batch, mode="sft"):
    """Weighted CE loss — Algorithm 1 line 6.
    Works for both MODEL_TYPE values since compute_per_example_ce handles branching.
    """
    input_ids      = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels         = batch["labels"].to(DEVICE)

    if mode == "sft":
      out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
      return out.loss, torch.ones(input_ids.size(0), device=DEVICE)

    weights   = batch["weight"].to(DEVICE)
    per_ce    = compute_per_example_ce(model, input_ids, attention_mask, labels)
    return (weights * per_ce).mean(), weights


print("ATLANTIS loss defined.")
print("Key: weights are pre-computed constants from frozen models.")

## 1.10 Model Helpers

In [ ]:
def _load_model(model_name, frozen=False):
    """Load T5 model — class depends on MODEL_TYPE."""
    if MODEL_TYPE == "classification":
        model = T5ForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(LABEL_LIST),
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
    else:
        model = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)

    if frozen:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
    return model

In [ ]:
def load_frozen_small_model():
    """Load flan-t5-small un-finetuned as p^S_b. Frozen — pre-computation only."""
    print(f"Loading p^S_b: {SMALL_MODEL} (frozen base)")
    model = _load_model(SMALL_MODEL, frozen=True)
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def load_frozen_large_model():
    """Load flan-t5-base un-finetuned as the frozen p^L_b.
    Used ONLY for pre-computing the p^L_b(y|x) term in the ATLANTIS weight.
    The model actually trained (p_θ) is loaded separately as a fresh copy.
    """
    print(f"Loading p^L_b frozen: {LARGE_MODEL} (frozen base, pre-computation only)")
    model = _load_model(LARGE_MODEL, frozen=True)
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def finetune_small_model(train_exs, tag="p_r", label_source="gold"):
    """Fine-tune flan-t5-small on train_exs to produce reference model p_r.
    Uses PR_EPOCHS (not NUM_EPOCHS) — p_r must converge before being used
    for weight computation. A poorly converged p_r produces noisy weights.
    """
    print(f"\nFine-tuning {tag} ({SMALL_MODEL}) for {PR_EPOCHS} epochs "
          f"on {len(train_exs)} examples...")
    model = _load_model(SMALL_MODEL)
    train_loader, _ = make_loaders(train_exs, label_source=label_source)

    opt   = torch.optim.AdamW(model.parameters(), lr=LR)
    total = len(train_loader) * PR_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)

    for epoch in range(PR_EPOCHS):
        model.train(); losses = []
        for batch in tqdm(train_loader, desc=f"  {tag} epoch {epoch+1}/{PR_EPOCHS}"):
            opt.zero_grad()
            out = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                labels=batch["labels"].to(DEVICE),
            )
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            losses.append(out.loss.item())
        print(f"  Epoch {epoch+1}  loss={np.mean(losses):.4f}")

    model.eval()
    print(f"{tag} fine-tuning complete.")
    return model


def compute_and_normalize_weights(examples, key="weight"):
    """Compute and self-normalize ATLANTIS weights for a list of examples.
    Each example must already have log_pr, log_ps_b, log_pl_b set.

    Normalization strategy depends on LOGPROB_MODE:
      'sum'  — logsumexp normalization in log space (numerically stable,
               avoids float32 underflow from large negative log-sum values)
      'mean' — simple arithmetic mean normalization (stable since mean
               log-probs are small, e.g. -0.5 to -3.0)

    In both cases the result has mean=1.0 so ATLANTIS loss is on the same
    scale as SFT.

    If WEIGHT_CLIP is set, weights are clipped then re-normalized.
    """
    from scipy.special import logsumexp

    log_w = np.array([ex["log_pl_b"] + ex["log_pr"] - ex["log_ps_b"]
                      for ex in examples])

    if LOGPROB_MODE == "sum":
        # log-space normalization — required when raw weights underflow to 0
        log_mean_w = logsumexp(log_w) - np.log(len(log_w))
        weights = np.exp(log_w - log_mean_w)
    else:
        # arithmetic normalization — stable with mean per-token log-probs
        raw_weights = np.exp(log_w)
        weights = raw_weights / raw_weights.mean()

    # Optional clipping
    if WEIGHT_CLIP is not None:
        weights = np.minimum(weights, WEIGHT_CLIP)
        weights = weights / weights.mean()   # re-normalize after clip

    for ex, w in zip(examples, weights):
        ex[key] = float(w)

    return weights


def train_large_model(train_exs, label_source="gold", include_weights=False,
                      loss_mode="sft", tag=""):
    """Train a fresh flan-t5-base (p_θ) and return (history, model).
    Weights are passed in as pre-computed constants via train_exs['weight'].
    """
    print(f"\n{'='*60}\n  {tag}\n{'='*60}")

    train_loader, test_loader = make_loaders(
        train_exs,
        label_source=label_source,
        include_weights=include_weights,
    )

    model = _load_model(LARGE_MODEL)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR)
    total = len(train_loader) * NUM_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)

    history = {"train_loss": [], "eval_f1": [], "weight_stats": []}

    for epoch in range(NUM_EPOCHS):
        model.train(); losses, all_w = [], []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
            opt.zero_grad()
            loss, w = atlantis_loss(model, batch, mode=loss_mode)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            losses.append(loss.item())
            all_w.extend(w.detach().cpu().tolist())

        avg_loss = np.mean(losses)
        f1, _, _ = evaluate(model, test_loader)

        w_arr   = np.array(all_w)
        w_stats = {
            "mean": float(w_arr.mean()), "std": float(w_arr.std()),
            "min":  float(w_arr.min()),  "max": float(w_arr.max()),
            "pct_clip_low":  0.0,  # no floor clipping
            "pct_clip_high": float((w_arr >= WEIGHT_CLIP - 1e-6).mean()) if WEIGHT_CLIP else 0.0,
        }
        history["train_loss"].append(avg_loss)
        history["eval_f1"].append(f1)
        history["weight_stats"].append(w_stats)

        line = f"  Epoch {epoch+1}  loss={avg_loss:.4f}  macro-F1={f1:.4f}"
        if loss_mode == "atlantis":
            line += f"  w_mean={w_stats['mean']:.3f}  w_std={w_stats['std']:.3f}"
        print(line)

    history["final_f1"] = history["eval_f1"][-1]
    print(f"  → Final F1: {history['final_f1']:.4f}")
    return history, model



print("Model helpers defined.")
print(f"p_r will train for PR_EPOCHS={PR_EPOCHS}, p^L_b for NUM_EPOCHS={NUM_EPOCHS}")

---
# Part 2 — Experiment

All models trained on the **same full clean dataset**. 


## 2.1 Fine-tune p_r on Full Training Set

In [ ]:
# p_r_A = flan-t5-small fine-tuned on full clean training data
p_r_A = finetune_small_model(train_all, tag="p_r_A")

In [ ]:
# Evaluate p_r on test set — should be reasonably high before using for weights
_, test_loader_pr = make_loaders(test_examples, label_source="gold")
f1_pr, _, _ = evaluate(p_r_A, test_loader_pr)   # or p_r_B for Paradigm B
print(f"p_r test F1: {f1_pr:.4f}")
print("p_r should achieve reasonable F1 before weight computation.")
print("If F1 < 0.3, increase PR_EPOCHS and retrain.")

## 2.2 Load p^S_b

In [ ]:
# p^S_b = flan-t5-small at init weights, frozen
p_s_b = load_frozen_small_model()

## 2.3 Cache Log-Probs

Compute `log p_r(y|x)` and `log p^S_b(y|x)` once for every training example.
Both are mean-per-token log-probs so they share the same scale as `log p^L_b`
computed live during training.

In [ ]:
print(f"Pre-computing ATLANTIS weights for {len(train_all)} examples...")
print(f"LOGPROB_MODE={LOGPROB_MODE!r}  WEIGHT_CLIP={WEIGHT_CLIP}")
print()

p_l_b_frozen = load_frozen_large_model()
train_A = [dict(ex) for ex in train_all]

for ex in tqdm(train_A, desc="Computing log-probs"):
    ex["log_pr"]   = compute_seq_logprob(p_r_A,        ex["input_text"], ex["label_str"])
    ex["log_ps_b"] = compute_seq_logprob(p_s_b,        ex["input_text"], ex["label_str"])
    ex["log_pl_b"] = compute_seq_logprob(p_l_b_frozen, ex["input_text"], ex["label_str"])

weights = compute_and_normalize_weights(train_A)

print(f"\nSample log_pr:   {train_A[0]['log_pr']:.3f}")
print(f"Sample log_ps_b: {train_A[0]['log_ps_b']:.3f}")
print(f"Sample log_pl_b: {train_A[0]['log_pl_b']:.3f}")
print()
print(f"Normalized weight mean: {weights.mean():.4f}  (always 1.0)")
print(f"Normalized weight std:  {weights.std():.4f}  (> 0 means ATLANTIS differs from SFT)")
print(f"Normalized weight min:  {weights.min():.4f}")
print(f"Normalized weight max:  {weights.max():.4f}")
print()

# Weight by relation type — check for class bias
by_label = {}
for ex in train_A:
    by_label.setdefault(ex["label_str"], []).append(ex["weight"])
print("Mean weight by relation type:")
for l, ws in sorted(by_label.items(), key=lambda x: -np.mean(x[1])):
    print(f"  {l:<25} mean={np.mean(ws):.3f}  std={np.std(ws):.3f}  n={len(ws)}")
print()
print("Healthy: weights should be roughly uniform across relation types.")
print("If one class dominates, consider switching LOGPROB_MODE or reducing WEIGHT_CLIP.")

## 2.4 Free Frozen Models

In [ ]:
del p_r_A, p_s_b, p_l_b_frozen
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 2.5 Run Paradigm A Experiments

In [ ]:
hist_A_sft, model_A_sft = train_large_model(
    train_A,
    label_source="gold",
    include_weights=False,
    loss_mode="sft",
    tag="Paradigm A — SFT baseline",
)

In [ ]:
del model_A_sft
if torch.cuda.is_available(): torch.cuda.empty_cache()

hist_A_atlantis, model_A_atlantis = train_large_model(
    train_A,
    label_source="gold",
    include_weights=True,
    loss_mode="atlantis",
    tag="Paradigm A — ATLANTIS",
)

In [ ]:
_, test_loader = make_loaders(train_A, label_source="gold")
_, preds, golds = evaluate(model_A_atlantis, test_loader)

from collections import Counter
print("Predicted distribution:")
for label, count in Counter(preds).most_common():
    print(f"  {label:<25} {count}")
print()
print("Gold distribution:")
for label, count in Counter(golds).most_common():
    print(f"  {label:<25} {count}")

## 2.6 Paradigm A Results

In [ ]:
results_A = {"SFT": hist_A_sft, "ATLANTIS": hist_A_atlantis}
COLORS_A  = {"SFT": "#e74c3c", "ATLANTIS": "#3498db"}
epochs_x  = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Paradigm A — Paper Replication (same clean dataset)",
             fontsize=13, fontweight="bold")

ax = axes[0]
for name, h in results_A.items():
    ax.plot(epochs_x, h["eval_f1"], marker="o", label=name, color=COLORS_A[name])
ax.set_title("Macro-F1 by Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Macro-F1")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for name, h in results_A.items():
    ax.plot(epochs_x, h["train_loss"], marker="s", label=name, color=COLORS_A[name])
ax.set_title("Training Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
names = list(results_A.keys())
f1s   = [results_A[n]["final_f1"] for n in names]
bars  = ax.bar(names, f1s, color=[COLORS_A[n] for n in names],
               edgecolor="black", linewidth=0.7, width=0.4)
ax.set_ylim(max(0, min(f1s) - 0.05), min(1.0, max(f1s) + 0.05))
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_title("Final Macro-F1"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("paradigm_A_results.png", dpi=150, bbox_inches="tight")
plt.show()

delta = hist_A_atlantis["final_f1"] - hist_A_sft["final_f1"]
print(f"SFT:       {hist_A_sft['final_f1']:.4f}")
print(f"ATLANTIS:  {hist_A_atlantis['final_f1']:.4f}")
print(f"Delta:     {delta:+.4f}")